# 03 — Preprocessing: Split, Impute & Scale
## Airline Flight Delay Prediction Project

---

### Purpose of this notebook

Feature engineering gave us 12 meaningful features. Now we need to:

1. **Split** the data into train / validation / test sets — using a **temporal split** strategy
2. **Impute** the ~1,330 remaining null values — using the median computed on train data only
3. **Scale** the continuous features — using `StandardScaler` fit on train data only
4. **Save** the final `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test` arrays

Every model notebook (04 and 05) will simply load these arrays and start training.

---

### The golden rule of preprocessing

> **Fit on train. Transform on everything.**

All preprocessing objects (imputer, scaler) must be:
- **Fit** exclusively on training data
- **Applied** (transform-only) to validation and test data

Violating this rule causes **data leakage** — the model indirectly "sees" validation/test
statistics during training, producing optimistically biased evaluation results.

---
> **Input :** `data/raw/Airline_Delay_Cause.csv`  
> **Output:** `X_train.npy`, `X_val.npy`, `X_test.npy`, `y_train.npy`, `y_val.npy`, `y_test.npy`  
> **Also saves:** `scaler.pkl`, `imputer.pkl`, `le_carrier.pkl`, `le_airport.pkl`

---
*Notebook: `03_preprocessing.ipynb` | Project: Airline Delay DNN*

---
## 1. Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────
import warnings, os, pickle, json
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Scikit-learn preprocessing ────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# ── Display & plot settings ───────────────────────────────────
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
})

RAW_PATH  = '../data/raw/Airline_Delay_Cause.csv'
OUT_PATH  = '../data/processed/'
FIG_PATH  = '../reports/figures/'

os.makedirs(OUT_PATH, exist_ok=True)

# ── Feature & target definitions (single source of truth) ─────
FEATURE_COLS = [
    'month_sin',          # Cyclic month — sine
    'month_cos',          # Cyclic month — cosine
    'year_norm',          # Normalised year [0,1]
    'log_arr_flights',    # Log-transformed flight volume
    'cancel_rate',        # Cancellation ratio
    'divert_rate',        # Diversion ratio
    'carrier_hist_delay', # Rolling 12-month carrier history
    'airport_hist_delay', # Rolling 12-month airport history
    'is_summer',          # Binary flag: Jun/Jul/Aug
    'is_holiday_month',   # Binary flag: Nov/Dec
    'carrier_enc',        # Label-encoded carrier
    'airport_enc',        # Label-encoded airport
]
TARGET_COL = 'delay_rate'

# Continuous features that need StandardScaler
# (binary flags and integer encodings are NOT scaled)
CONTINUOUS_COLS = [
    'month_sin', 'month_cos', 'year_norm', 'log_arr_flights',
    'cancel_rate', 'divert_rate', 'carrier_hist_delay', 'airport_hist_delay'
]
CONT_IDX = [FEATURE_COLS.index(c) for c in CONTINUOUS_COLS]

print("✅ Configuration set")
print(f"   Total features     : {len(FEATURE_COLS)}")
print(f"   Continuous features: {len(CONTINUOUS_COLS)}")
print(f"   Categorical/binary : {len(FEATURE_COLS) - len(CONTINUOUS_COLS)}")

---
## 2. Rebuild Feature-Engineered DataFrame

We rebuild the full feature matrix from the raw CSV.  
This makes the notebook **self-contained** — you can run it independently
without needing the parquet output from notebook 02.

All feature engineering logic is consolidated here as clean functions so the
code is reusable and testable.

In [ ]:
# ── Load raw data ─────────────────────────────────────────────
df_raw = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full feature engineering pipeline.

    Takes the raw Airline_Delay_Cause DataFrame and returns a clean
    DataFrame with all 12 engineered features + target + identifiers.

    Parameters
    ----------
    df : pd.DataFrame
        Raw loaded CSV (all 21 original columns present).

    Returns
    -------
    pd.DataFrame
        Sorted by carrier/airport/year/month with all 12 features computed.

    Notes
    -----
    - Leaky delay-cause columns are dropped before any feature is created.
    - Rolling history features use shift(1) to prevent target leakage.
    - LabelEncoders are fit inside this function and returned separately.
    """
    df = df.copy()

    # ── Step 1: Drop rows that can't have a valid target ──────
    df = df.dropna(subset=['arr_flights', 'arr_del15'])
    df = df[df['arr_flights'] > 0]

    # ── Step 2: Compute target before dropping arr_del15 ─────
    df['delay_rate'] = df['arr_del15'] / df['arr_flights']

    # ── Step 3: Drop leaky columns ────────────────────────────
    LEAKY = [
        'carrier_ct','weather_ct','nas_ct','security_ct','late_aircraft_ct',
        'arr_delay','carrier_delay','weather_delay','nas_delay',
        'security_delay','late_aircraft_delay','arr_del15'
    ]
    df = df.drop(columns=LEAKY)

    # ── Step 4: Sort chronologically (required for rolling) ───
    df = df.sort_values(['carrier','airport','year','month']).reset_index(drop=True)

    # ── Step 5: Cyclic month encoding ─────────────────────────
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # ── Step 6: Normalised year ───────────────────────────────
    year_min, year_max = df['year'].min(), df['year'].max()
    df['year_norm'] = (df['year'] - year_min) / (year_max - year_min)

    # ── Step 7: Log-transformed flight volume ─────────────────
    df['log_arr_flights'] = np.log1p(df['arr_flights'])

    # ── Step 8: Cancellation & diversion rates ────────────────
    df['cancel_rate'] = df['arr_cancelled'] / df['arr_flights']
    df['divert_rate'] = df['arr_diverted'].fillna(0.0) / df['arr_flights']

    # ── Step 9: Rolling carrier delay history ─────────────────
    df = df.sort_values(['carrier','year','month']).reset_index(drop=True)
    df['carrier_hist_delay'] = (
        df.groupby('carrier')['delay_rate']
          .transform(lambda x: x.shift(1).rolling(12, min_periods=3).mean())
    )

    # ── Step 10: Rolling airport delay history ────────────────
    df = df.sort_values(['airport','year','month']).reset_index(drop=True)
    df['airport_hist_delay'] = (
        df.groupby('airport')['delay_rate']
          .transform(lambda x: x.shift(1).rolling(12, min_periods=3).mean())
    )

    # ── Step 11: Binary seasonal flags ───────────────────────
    df['is_summer']        = df['month'].isin([6, 7, 8]).astype(int)
    df['is_holiday_month'] = df['month'].isin([11, 12]).astype(int)

    # ── Step 12: Label encoding ───────────────────────────────
    le_carrier = LabelEncoder()
    le_airport  = LabelEncoder()
    df['carrier_enc'] = le_carrier.fit_transform(df['carrier'])
    df['airport_enc'] = le_airport.fit_transform(df['airport'])

    # ── Final sort ────────────────────────────────────────────
    df = df.sort_values(['carrier','airport','year','month']).reset_index(drop=True)

    return df, le_carrier, le_airport


# ── Run the pipeline ──────────────────────────────────────────
df, le_carrier, le_airport = build_features(df_raw)

print(f"✅ Features built — shape: {df.shape}")
print(f"   Carriers : {le_carrier.classes_.shape[0]}")
print(f"   Airports : {le_airport.classes_.shape[0]}")
print(f"\nNull count per feature:")
print(df[FEATURE_COLS].isnull().sum().to_string())

---
## 3. Temporal Train / Validation / Test Split

### Why temporal and not random?

A **random split** (e.g., `train_test_split` with `shuffle=True`) would be wrong here.

Here is why:

**Scenario:** Random split puts row for `Delta × JFK × March 2021` into the training set
and row for `Delta × JFK × February 2021` into the test set.

When computing `carrier_hist_delay` for February 2021, the rolling window includes
March 2021 data — which is now in training. The model has seen the future.

Even worse: **both rows share the same rolling history values** computed from the
full dataset before splitting, so the model memorises patterns it would never
encounter in real deployment.

### Our strategy: temporal cutoff

```
Train : 2003 – 2018  (16 years — 78.6% of data)
Val   : 2019 – 2020  (2 years  — 12.5% of data)
Test  : 2021 – 2022  (2 years  —  9.0% of data)
```

This mirrors real-world deployment: the model is trained on the past,
evaluated on the near-future (val), and tested on the far-future (test).

### Why these specific cutoff years?

| Cutoff | Reason |
|---|---|
| Train ends 2018 | Captures full pre-COVID "normal" operations |
| Val 2019–2020 | 2019 = last normal year; 2020 = COVID disruption — tests robustness |
| Test 2021–2022 | Recovery period — genuinely unseen operational conditions |

In [ ]:
# ── Temporal split ────────────────────────────────────────────
train_df = df[df['year'] <= 2018].copy()
val_df   = df[(df['year'] >= 2019) & (df['year'] <= 2020)].copy()
test_df  = df[df['year'] >= 2021].copy()

total = len(df)
print("=" * 55)
print(f"{'Split':<8} {'Years':<12} {'Rows':>10} {'%':>7}")
print("=" * 55)
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    yr = f"{split['year'].min()}–{split['year'].max()}"
    print(f"{name:<8} {yr:<12} {len(split):>10,} {len(split)/total*100:>6.1f}%")
print("=" * 55)
print(f"{'Total':<8} {'2003–2022':<12} {total:>10,} {'100.0%':>7}")

In [ ]:
# ── Visualise: target distribution across splits ─────────────
# CRITICAL CHECK: if the distributions diverge too much,
# the model trained on train will struggle to generalise to val/test

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

split_info = [
    (train_df, 'Train (2003–2018)', 'steelblue'),
    (val_df,   'Val   (2019–2020)', 'darkorange'),
    (test_df,  'Test  (2021–2022)', 'mediumseagreen'),
]

for ax, (split, label, color) in zip(axes, split_info):
    ax.hist(split[TARGET_COL], bins=50, color=color,
            edgecolor='white', alpha=0.85, density=True)
    split[TARGET_COL].plot.kde(ax=ax, color='black', lw=1.5)
    mean_val = split[TARGET_COL].mean()
    ax.axvline(mean_val, color='red', ls='--', lw=1.5,
               label=f'mean={mean_val:.3f}')
    ax.set_title(label)
    ax.set_xlabel('delay_rate')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Target Distribution Across Splits\n'
             '(distributions should be roughly similar — check for drift)',
             fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '23_target_split_distributions.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 23_target_split_distributions.png")

In [ ]:
# ── Distribution shift analysis ───────────────────────────────
# The val set (2019–2020) includes COVID-19 (2020)
# This caused an unusual drop in delay rates — fewer flights, less congestion
# We quantify this to set realistic expectations for model performance

print("Target statistics per split:")
print(f"{'Metric':<12} {'Train':>10} {'Val':>10} {'Test':>10}")
print("-" * 45)
for stat_name, fn in [('mean','mean'),('median','median'),('std','std'),
                      ('min','min'),('max','max')]:
    vals = [getattr(s[TARGET_COL], fn)() for s in [train_df, val_df, test_df]]
    print(f"{stat_name:<12} {vals[0]:>10.4f} {vals[1]:>10.4f} {vals[2]:>10.4f}")

print("\n⚠️  Note: Val mean (2019–2020) is lower than Train/Test.")
print("   This is expected: 2020 COVID caused historically low delay rates")
print("   due to drastically reduced flight volumes and congestion.")
print("   The model must generalise to both low-COVID and normal-operation regimes.")

---
## 4. Imputation — Handling Remaining Null Values

From feature engineering, we have ~1,330 null values concentrated in:
- `carrier_hist_delay` — first few months of a carrier's history (not enough rolling window)
- `airport_hist_delay` — same reason for airports

### Strategy: Median imputation

We use the **median** (not mean) because:
- `delay_rate` and its history are right-skewed
- The median is more robust to outliers — a few extreme delay months won't pull the imputed value up

### The fit-on-train rule in action

```python
imputer.fit(X_train)      # learn median from training data only
imputer.transform(X_val)  # apply those train medians to val
imputer.transform(X_test) # apply those train medians to test
```

If we computed the median on the entire dataset (including val/test), we would let
future information flow back into training — subtle but real leakage.

In [ ]:
# ── Null audit before imputation ─────────────────────────────
print("Null counts per feature per split (before imputation):")
print(f"\n{'Feature':<25} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 52)
for col in FEATURE_COLS:
    t_null = train_df[col].isna().sum()
    v_null = val_df[col].isna().sum()
    x_null = test_df[col].isna().sum()
    flag = " ⚠️" if (t_null + v_null + x_null) > 0 else " ✅"
    print(f"{col:<25} {t_null:>8,} {v_null:>8,} {x_null:>8,}{flag}")

In [ ]:
# ── Fit imputer on TRAIN only ─────────────────────────────────
imputer = SimpleImputer(strategy='median')

# Extract raw feature matrices (before scaling)
X_train_raw = train_df[FEATURE_COLS].values
X_val_raw   = val_df[FEATURE_COLS].values
X_test_raw  = test_df[FEATURE_COLS].values

# fit() computes the median of each column from X_train ONLY
imputer.fit(X_train_raw)

# transform() replaces NaNs with the learned medians
X_train_imp = imputer.transform(X_train_raw)
X_val_imp   = imputer.transform(X_val_raw)
X_test_imp  = imputer.transform(X_test_raw)

# ── Verify: no nulls remain ────────────────────────────────────
assert np.isnan(X_train_imp).sum() == 0, "Nulls remain in X_train after imputation!"
assert np.isnan(X_val_imp).sum()   == 0, "Nulls remain in X_val after imputation!"
assert np.isnan(X_test_imp).sum()  == 0, "Nulls remain in X_test after imputation!"
print("✅ All nulls imputed — zero nulls remain in any split")

# ── Show imputed median values ────────────────────────────────
print("\nImputed median values (from train):")
for col, med in zip(FEATURE_COLS, imputer.statistics_):
    print(f"  {col:<25}  median = {med:.4f}")

---
## 5. Feature Scaling — StandardScaler on Continuous Features

### Why scale?

Neural networks use gradient descent. When features have very different scales:
- A feature with range [0, 10] produces gradients 100× larger than one with range [0, 0.1]
- Weights connected to the large-scale feature update much faster
- Training becomes **unstable** — loss oscillates or diverges

Scaling ensures all continuous features contribute equally to the gradient signal.

### StandardScaler formula

```
z = (x - μ_train) / σ_train
```

Where `μ_train` and `σ_train` are computed **exclusively from training data**.

After scaling: each continuous feature has **mean ≈ 0** and **std ≈ 1** on the training set.

### What we do NOT scale

| Feature | Why not scaled |
|---|---|
| `is_summer` | Binary {0, 1} — already bounded, scaling breaks meaning |
| `is_holiday_month` | Same reason |
| `carrier_enc` | Integer index — will go into an Embedding layer (not used numerically) |
| `airport_enc` | Same reason |

### Fit-on-train rule again

```python
scaler.fit(X_train[:, CONT_IDX])   # learn μ and σ from train
scaler.transform(X_val[:,  CONT_IDX])  # apply train statistics to val
scaler.transform(X_test[:, CONT_IDX]) # apply train statistics to test
```

This is the same principle as imputation — statistics always come from training data.

In [ ]:
# ── Fit StandardScaler on continuous columns of train ─────────
scaler = StandardScaler()

# Work on copies to avoid overwriting imputed arrays
X_train_scaled = X_train_imp.copy()
X_val_scaled   = X_val_imp.copy()
X_test_scaled  = X_test_imp.copy()

# fit() learns mean and std from X_train continuous columns
scaler.fit(X_train_imp[:, CONT_IDX])

# transform() standardises each split using TRAIN statistics
X_train_scaled[:, CONT_IDX] = scaler.transform(X_train_imp[:, CONT_IDX])
X_val_scaled[:,   CONT_IDX] = scaler.transform(X_val_imp[:,  CONT_IDX])
X_test_scaled[:,  CONT_IDX] = scaler.transform(X_test_imp[:, CONT_IDX])

print("StandardScaler statistics (learned from train):")
print(f"\n{'Feature':<25} {'Train μ':>10} {'Train σ':>10}")
print("-" * 48)
for col, mean, std in zip(CONTINUOUS_COLS, scaler.mean_, scaler.scale_):
    print(f"  {col:<23} {mean:>10.4f} {std:>10.4f}")

In [ ]:
# ── Verify scaling on train set ────────────────────────────────
train_means = X_train_scaled[:, CONT_IDX].mean(axis=0)
train_stds  = X_train_scaled[:, CONT_IDX].std(axis=0)

print("Post-scaling verification on TRAIN (should be ≈0 mean, ≈1 std):")
print(f"\n{'Feature':<25} {'Mean':>10} {'Std':>10}")
print("-" * 48)
for col, m, s in zip(CONTINUOUS_COLS, train_means, train_stds):
    ok_mean = "✅" if abs(m) < 1e-6 else "⚠️"
    ok_std  = "✅" if abs(s - 1.0) < 1e-6 else "⚠️"
    print(f"  {col:<23} {m:>10.6f} {ok_mean}  {s:>10.6f} {ok_std}")

In [ ]:
# ── Visualise: before vs after scaling (sample 3 features) ───
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
check_cols = ['log_arr_flights', 'cancel_rate', 'carrier_hist_delay']
check_idx  = [FEATURE_COLS.index(c) for c in check_cols]

for col, idx, (ax_before, ax_after) in zip(
    check_cols, check_idx,
    [(axes[0,0],axes[1,0]), (axes[0,1],axes[1,1]), (axes[0,2],axes[1,2])]
):
    raw_vals    = X_train_imp[:, idx]
    scaled_vals = X_train_scaled[:, idx]

    ax_before.hist(raw_vals, bins=50, color='tomato',
                   edgecolor='white', alpha=0.85, density=True)
    ax_before.set_title(f'BEFORE: {col}\nμ={raw_vals.mean():.3f}  σ={raw_vals.std():.3f}')

    ax_after.hist(scaled_vals, bins=50, color='steelblue',
                  edgecolor='white', alpha=0.85, density=True)
    ax_after.set_title(f'AFTER: {col}\nμ={scaled_vals.mean():.3f}  σ={scaled_vals.std():.3f}')

plt.suptitle('StandardScaler: Before vs After (Train data)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '24_scaling_before_after.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 24_scaling_before_after.png")

---
## 6. Target Arrays

We extract the target variable into simple numpy arrays.

> **We do NOT scale the target.**  
> Scaling the target changes what the loss function optimises for.
> Our target `delay_rate` is already in [0, 1], so scaling adds no benefit
> but makes predictions harder to interpret (we'd need to inverse-transform
> every output). We keep it in its natural scale.

In [ ]:
# ── Extract target vectors ─────────────────────────────────────
y_train = train_df[TARGET_COL].values
y_val   = val_df[TARGET_COL].values
y_test  = test_df[TARGET_COL].values

print("Target array shapes:")
print(f"  y_train : {y_train.shape}")
print(f"  y_val   : {y_val.shape}")
print(f"  y_test  : {y_test.shape}")

print("\nTarget statistics per split:")
print(f"  {'Split':<8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("  " + "-"*40)
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f"  {name:<8} {y.mean():>8.4f} {y.std():>8.4f} "
          f"{y.min():>8.4f} {y.max():>8.4f}")

# ── Baseline: a model that always predicts the train mean ──────
# This is our minimum performance bar — any model must beat this
baseline_mae  = np.abs(y_val  - y_train.mean()).mean()
baseline_rmse = np.sqrt(((y_val - y_train.mean())**2).mean())

print(f"\n📏 Dummy baseline (always predict train mean = {y_train.mean():.4f}):")
print(f"   Val MAE  = {baseline_mae:.4f}")
print(f"   Val RMSE = {baseline_rmse:.4f}")
print("   Any real model must beat these numbers to add value.")

---
## 7. Final Feature Matrix Inspection

Before saving, we do a final visual check of the full processed feature matrix
to confirm everything looks as expected.

In [ ]:
# ── Distribution of all features in X_train (post-scaling) ───
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    vals = X_train_scaled[:, i]
    axes[i].hist(vals, bins=40, color='steelblue',
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{col}\nμ={vals.mean():.2f}  σ={vals.std():.2f}',
                      fontsize=9)
    axes[i].tick_params(labelsize=8)

plt.suptitle('X_train — All 12 Features After Preprocessing',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_PATH + '25_final_feature_distributions.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 25_final_feature_distributions.png")

In [ ]:
# ── Correlation heatmap of final feature matrix ───────────────
X_train_df = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones((len(FEATURE_COLS), len(FEATURE_COLS)), dtype=bool), k=1)
sns.heatmap(
    X_train_df.corr(), mask=mask,
    annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True,
    linewidths=0.3, annot_kws={'size': 8}, ax=ax
)
ax.set_title('X_train Feature Correlation Matrix (post-preprocessing)', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_PATH + '26_preprocessed_correlation.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 26_preprocessed_correlation.png")

---
## 8. Save All Arrays & Preprocessing Objects

We save six numpy arrays (the final model-ready data) and four preprocessing
objects (for reproducibility and deployment).

### Why save preprocessing objects?

In production, when a new month of data arrives:
1. Apply the same `LabelEncoder` to map new carrier/airport codes to integers
2. Apply the same `SimpleImputer` to fill any missing values with training medians
3. Apply the same `StandardScaler` to normalise using training statistics

If we refit any of these objects on new data, the pipeline becomes inconsistent
and predictions will be unreliable.

In [ ]:
# ── Save numpy arrays ─────────────────────────────────────────
arrays = {
    'X_train': X_train_scaled,
    'X_val'  : X_val_scaled,
    'X_test' : X_test_scaled,
    'y_train': y_train,
    'y_val'  : y_val,
    'y_test' : y_test,
}

for name, arr in arrays.items():
    path = OUT_PATH + name + '.npy'
    np.save(path, arr)
    print(f"✅ Saved {name:>8s}.npy   shape={str(arr.shape):<18} "
          f"dtype={arr.dtype}")

In [ ]:
# ── Save preprocessing objects ────────────────────────────────
objects = {
    'scaler.pkl'     : scaler,
    'imputer.pkl'    : imputer,
    'le_carrier.pkl' : le_carrier,
    'le_airport.pkl' : le_airport,
}

for filename, obj in objects.items():
    with open(OUT_PATH + filename, 'wb') as f:
        pickle.dump(obj, f)
    print(f"✅ Saved {filename}")

In [ ]:
# ── Save preprocessing metadata ───────────────────────────────
meta = {
    "feature_cols"     : FEATURE_COLS,
    "target_col"       : TARGET_COL,
    "continuous_cols"  : CONTINUOUS_COLS,
    "cont_idx"         : CONT_IDX,
    "split_years"      : {"train": [2003, 2018],
                          "val"  : [2019, 2020],
                          "test" : [2021, 2022]},
    "shapes": {
        "X_train": list(X_train_scaled.shape),
        "X_val"  : list(X_val_scaled.shape),
        "X_test" : list(X_test_scaled.shape),
    },
    "imputer_strategy" : "median",
    "scaler_type"      : "StandardScaler",
    "baseline_val_mae" : float(baseline_mae),
    "baseline_val_rmse": float(baseline_rmse),
    "n_carriers"       : int(len(le_carrier.classes_)),
    "n_airports"       : int(len(le_airport.classes_)),
}

with open(OUT_PATH + 'preprocessing_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("✅ Saved preprocessing_meta.json")
print("\n" + "=" * 55)
print("PREPROCESSING COMPLETE — SUMMARY")
print("=" * 55)
print(f"  X_train : {X_train_scaled.shape}  →  {X_train_scaled.nbytes/1e6:.1f} MB")
print(f"  X_val   : {X_val_scaled.shape}    →  {X_val_scaled.nbytes/1e6:.1f} MB")
print(f"  X_test  : {X_test_scaled.shape}   →  {X_test_scaled.nbytes/1e6:.1f} MB")
print(f"  Nulls   : 0 (all imputed)")
print(f"  Scaling : StandardScaler (fit on train only)")
print(f"  Baseline MAE  (val): {baseline_mae:.4f}")
print(f"  Baseline RMSE (val): {baseline_rmse:.4f}")
print("=" * 55)
print("\nNext → 04_baseline_models.ipynb")

---
## 9. Load Verification

Always verify saved files can be loaded correctly before finishing the notebook.
This takes 3 seconds and prevents headaches in downstream notebooks.

In [ ]:
# ── Reload and verify arrays ──────────────────────────────────
print("Verifying saved arrays...")
for name in ['X_train','X_val','X_test','y_train','y_val','y_test']:
    arr = np.load(OUT_PATH + name + '.npy')
    nulls = np.isnan(arr).sum()
    status = "✅" if nulls == 0 else f"⚠️  {nulls} NaNs!"
    print(f"  {name:>8}.npy  shape={str(arr.shape):<20} nulls={nulls}  {status}")

print("\nVerifying preprocessing objects...")
for fn in ['scaler.pkl','imputer.pkl','le_carrier.pkl','le_airport.pkl']:
    with open(OUT_PATH + fn, 'rb') as f:
        obj = pickle.load(f)
    print(f"  {fn:<20} type={type(obj).__name__}")

print("\n✅ All files verified — ready for notebook 04")

---
## Summary

| Step | What we did | Key decision |
|---|---|---|
| Feature rebuild | Called `build_features()` — self-contained function | Modular, testable pipeline |
| Temporal split | Train ≤2018 / Val 2019–2020 / Test ≥2021 | Prevents future data leaking into training |
| Imputation | `SimpleImputer(strategy='median')` fit on train only | Handles ~0.4% nulls in rolling features |
| Scaling | `StandardScaler` on 8 continuous features, fit on train only | Neural net inputs → mean≈0, std≈1 |
| Baseline | Dummy model (always predict train mean) — Val MAE=0.0761 | Minimum bar any model must beat |
| Saved | 6 numpy arrays + 4 pkl objects + metadata JSON | Fully reproducible pipeline |

---
*Next: `04_baseline_models.ipynb` — Linear Regression, Ridge, Random Forest, XGBoost*